# Psychology Quality Dataset Pipeline

This notebook downloads and processes psychology-related conversational data from multiple Hugging Face datasets. It transforms the data into a unified format, removes duplicates, and applies quality and safety filters to create a high-quality training dataset for psychology-focused AI models.

## Pipeline Overview
- **Step 1**: Install required packages (PyTorch, Unsloth, datasets, etc.)
- **Step 2**: Define datasets and helper functions for format conversion
- **Step 3**: Load and process multiple psychology datasets
- **Step 4**: Merge all data and remove duplicates
- **Step 5**: Apply quality and safety filters
- **Step 6**: Save final dataset as CSV and JSON


## 1. Package Installation

Install all required libraries for:
- **PyTorch** with CUDA 12.4 support (GPU acceleration)
- **Unsloth** - Efficient fine-tuning library
- **Hugging Face datasets** - For loading and processing datasets
- **Google BigQuery** - For cloud data operations


In [ ]:
!pip install pandas
!pip install datasets

## 2. Import Libraries

Import necessary libraries for data processing:
- `datasets` - Loading Hugging Face datasets
- `pandas` - Data manipulation
- `json` - JSON file handling
- `re` - Regular expressions for text parsing
- `os`, `shutil` - File system operations


In [67]:
# Import libraries
from datasets import load_dataset
import pandas as pd
import json
import re
import os
import shutil

print("✅ Libraries installed and imported")

✅ Libraries installed and imported


## 3. Dataset Configuration

Define the Hugging Face datasets to be processed:

| Dataset | Description |
|---------|-------------|
| `Gragroo/psy_conv_v1_gpt1/2/3` | Psychology conversations (GPT format) |
| `psycode1/psyset` | Psychology dataset with INST format |
| `Gragroo/psychology-question-answer_psygpt_format` | Q&A format psychology data |


In [68]:
# Define datasets exactly as you specified
DATASETone = "Gragroo/psy_conv_v1_gpt1"
DATASETtwo = "Gragroo/psy_conv_v1_gpt2"
DATASETthree = "Gragroo/psy_conv_v1_gpt3"

alldataCATone = []

datasetLIST = [DATASETone, DATASETtwo, DATASETthree]

# Dataset 4: psycode1/psyset
datasetCATtwo = "psycode1/psyset"

# Dataset 5: Gragroo/psychology-question-answer_psygpt_format
datasetCATthree = "Gragroo/psychology-question-answer_psygpt_format"

print("✅ Datasets defined:")
print(f"   - {DATASETone}")
print(f"   - {DATASETtwo}")
print(f"   - {DATASETthree}")
print(f"   - {datasetCATtwo}")
print(f"   - {datasetCATthree}")

✅ Datasets defined:
   - Gragroo/psy_conv_v1_gpt1
   - Gragroo/psy_conv_v1_gpt2
   - Gragroo/psy_conv_v1_gpt3
   - psycode1/psyset
   - Gragroo/psychology-question-answer_psygpt_format


## 4. Helper Functions

Three conversion functions handle different data formats:

### 4.1 `convert_json_format()`
Converts JSON conversation lists to Alpaca format with 'user' and 'assistant' roles.

### 4.2 `extract_inst_format()`
Parses `[INST] instruction [/INST] response` format from text fields.

### 4.3 `extract_qa_format()`
Extracts question/answer pairs from structured fields (question/answer or instruction/response).


In [69]:
# Function 1: Convert JSON conversation list to Alpaca format
def convert_json_format(conversations):
    alpaca = []
    for msg in conversations:
        role = msg.get("from", "").strip().lower()
        content = msg.get("value", "")
        
        if role in ["human", "user"]:
            alpaca.append({"role": "user", "content": content})
        elif role in ["gpt", "assistant", "ai"]:
            alpaca.append({"role": "assistant", "content": content})
    return alpaca


# Function 2: Extract from [INST] text format
def extract_inst_format(text):
    """Handle datasets with format: '<s>[INST] instruction [/INST] response'"""
    if pd.isna(text) or not isinstance(text, str):
        return None, None
    
    pattern = r'<s>\\[INST\\](.*?)\\[/INST\\]\\s*(.+?)\\s*(?:<|nan|$)'
    match = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
    
    if match:
        return match.group(1).strip(), match.group(2).strip()
    
    alt_pattern = r'\\[INST\\](.+?)\\[/INST\\]'
    matches = re.findall(alt_pattern, text, re.DOTALL | re.IGNORECASE)
    
    if matches:
        instruction = matches[0].strip()
        remaining = text.split('[/INST]')[-1].strip()
        response = re.sub(r'\\s*<.*', '', remaining).strip()
        if instruction and response:
            return instruction, response
    
    return None, None


# Function 3: Extract from question/answer format
def extract_qa_format(row):
    """Handle datasets with question/answer fields"""
    question = row.get('question') or row.get('instruction')
    answer = row.get('answer') or row.get('response')
    
    if question and answer:
        convs = [
            {"role": "user", "content": str(question)},
            {"role": "assistant", "content": str(answer)}
        ]
        return convs
    return None

print("✅ Helper functions defined")

✅ Helper functions defined


## 5. Process Datasets 1-3

Load and process datasets with conversation format (GPT-style conversations). Each dataset contains message lists that are converted to unified Alpaca format.

**Output**: List of conversation objects with 'user' and 'assistant' messages


In [70]:
# Process first 3 datasets (conversations format)
def datasetCatONEmerge(datasetName):
    print(f"📥 Loading: {datasetName}")
    datasetCatONE = load_dataset(datasetName, split="train")
    print(f"   ✅ Loaded {len(datasetCatONE)} rows")

    dataTEMP = []
    for row in datasetCatONE:
        try:
            convs = row.get('conversations') or row.get('message') or row.get('chat')
            if convs:
                alpaca = convert_json_format(convs)
                dataTEMP.append({"conversations": alpaca, "dataset_name": datasetName})
        except:
            continue

    print(f"   ✅ Processed: {len(dataTEMP)} conversations")
    return dataTEMP


for dataset in datasetLIST:
    dataTEMP = datasetCatONEmerge(dataset)
    alldataCATone = alldataCATone + dataTEMP

print(f"\n📊 Dataset 1-3 Total: {len(alldataCATone)} conversations")

📥 Loading: Gragroo/psy_conv_v1_gpt1
   ✅ Loaded 102834 rows
   ✅ Processed: 102834 conversations
📥 Loading: Gragroo/psy_conv_v1_gpt2
   ✅ Loaded 102834 rows
   ✅ Processed: 102834 conversations
📥 Loading: Gragroo/psy_conv_v1_gpt3
   ✅ Loaded 102835 rows
   ✅ Processed: 102835 conversations

📊 Dataset 1-3 Total: 308503 conversations


## 6. Process Dataset 4

Load `psycode1/psyset` which uses INST text format. The text contains `[INST]` and `[/INST]` tags with instructions and responses embedded in a single text field.

**Processing**: Extract instruction and response using regex pattern matching


In [71]:
# Dataset 4: psycode1/psyset (INST text format)
print(f"📥 Loading: {datasetCATtwo}")
dataset2 = load_dataset(datasetCATtwo, split="train")
print(f"   ✅ Loaded {len(dataset2)} rows")

alldataCATtwo = []
for row in dataset2:
    try:
        text = row.get('text') or row.get('content')
        if text:
            instruction, response = extract_inst_format(str(text))
            if instruction and response:
                convs = [
                    {"role": "user", "content": instruction},
                    {"role": "assistant", "content": response}
                ]
                alldataCATtwo.append({"conversations": convs, "dataset_name": datasetCATtwo})
    except:
        continue

print(f"   ✅ Processed: {len(alldataCATtwo)} conversations")

📥 Loading: psycode1/psyset
   ✅ Loaded 8199 rows
   ✅ Processed: 0 conversations


## 7. Process Dataset 5

Load `Gragroo/psychology-question-answer_psygpt_format` which has structured question/answer fields. This is the simplest format to process.

**Processing**: Directly extract 'question' and 'answer' fields


In [72]:
# Dataset 5: Gragroo/psychology-question-answer_psygpt_format (question/answer format)
print(f"📥 Loading: {datasetCATthree}")
dataset3 = load_dataset(datasetCATthree, split="train")
print(f"   ✅ Loaded {len(dataset3)} rows")

alldataCATthree = []
for row in dataset3:
    try:
        convs = extract_qa_format(row)
        if convs:
            alldataCATthree.append({"conversations": convs, "dataset_name": datasetCATthree})
    except:
        continue

print(f"   ✅ Processed: {len(alldataCATthree)} conversations")

📥 Loading: Gragroo/psychology-question-answer_psygpt_format
   ✅ Loaded 197180 rows
   ✅ Processed: 197180 conversations


## 8. Merge All Data

Combine all processed datasets into a single list. Display summary statistics showing the count from each source dataset.

**Purpose**: Create a unified dataset with all psychology conversations


In [73]:
# Merge all data
alldataMERGE = alldataCATone + alldataCATtwo + alldataCATthree

print("=" * 50)
print("📊 MERGE SUMMARY")
print("=" * 50)
print(f"Dataset 1 ({DATASETone}): {sum(1 for item in alldataCATone if item.get('dataset_name') == DATASETone)}")
print(f"Dataset 2 ({DATASETtwo}): {sum(1 for item in alldataCATone if item.get('dataset_name') == DATASETtwo)}")
print(f"Dataset 3 ({DATASETthree}): {sum(1 for item in alldataCATone if item.get('dataset_name') == DATASETthree)}")
print(f"Dataset 4 ({datasetCATtwo}): {len(alldataCATtwo)}")
print(f"Dataset 5 ({datasetCATthree}): {len(alldataCATthree)}")
print(f"Total merged: {len(alldataMERGE)}")

📊 MERGE SUMMARY
Dataset 1 (Gragroo/psy_conv_v1_gpt1): 102834
Dataset 2 (Gragroo/psy_conv_v1_gpt2): 102834
Dataset 3 (Gragroo/psy_conv_v1_gpt3): 102835
Dataset 4 (psycode1/psyset): 0
Dataset 5 (Gragroo/psychology-question-answer_psygpt_format): 197180
Total merged: 505683


## 9. Deduplication

Remove duplicate conversation pairs. Duplicates are identified by identical (user_message, assistant_message) tuples.

**Why**: Prevents overfitting during model training caused by repeated examples


In [74]:
# Remove duplicates - keep only unique (user, assistant) pairs
print("\n🧹 Removing duplicates...")
print("   (If user msg + assistant msg are identical, remove duplicate)")

seen = set()
unique_rows = []

for item in alldataMERGE:
    convs = item.get('conversations', [])
    user_msg = next((m['content'] for m in convs if m['role'] == 'user'), "")
    assistant_msg = next((m['content'] for m in convs if m['role'] == 'assistant'), "")
    
    # Create key from (user_message, assistant_message)
    key = (user_msg, assistant_msg)
    if key not in seen:
        seen.add(key)
        unique_rows.append(item)

duplicates_removed = len(alldataMERGE) - len(unique_rows)
print(f"   Removed {duplicates_removed} duplicates")
print(f"   Unique rows: {len(unique_rows)}")


🧹 Removing duplicates...
   (If user msg + assistant msg are identical, remove duplicate)
   Removed 197106 duplicates
   Unique rows: 308577


## 10. Quality & Safety Filters

### Quality Filter
Keeps only responses containing therapeutic/psychology keywords (e.g., 'understand', 'feel', 'therapy', 'support', 'coping').

### Safety Filter
Removes responses containing harmful content keywords (e.g., 'suicide', 'violence', 'abuse', 'harm').

**Purpose**: Ensure the dataset contains helpful, safe psychology conversations suitable for training


In [75]:
# Quality & Safety Filters
# Words that indicate high-quality therapeutic responses
quality_words = [
    "understand", "feel", "feelings", "help", "support", "anxiety", "stress",
    "depression", "therapy", "therapist", "coping", "emotion", "emotional",
    "comfort", "validate", "validation", "heal", "healing", "hope", "listen",
    "listening", "care", "caring", "concern", "concerned", "understand", "understood",
    "relationship", "relationships", "family", "friends", "communication",
    "self-esteem", "confidence", "mental", "mental health", "well-being", "wellbeing",
    "mindful", "mindfulness", "relax", "relaxation", "calm", "peace", "peaceful",
    "trauma", "traumatic", "grief", "loss", "sad", "sadness", "lonely", "loneliness",
    "overwhelmed", "burden", "stressed", "worry", "worried", "fear", "fearful",
    "panic", "anger", "frustrated", "guilt", "shame", "embarrassed", "hurt", "pain",
    "struggle", "difficult", "challenge", "coping", "manage", "treatment", "heal",
    "recovery", "progress", "growth", "personal", "development", "awareness",
    "insight", "reflection", "thoughts", "beliefs", "behaviors", "patterns", "habits",
    "boundaries", "respect", "trust", "honest", "open", "vulnerable", "share",
    "sharing", "process", "journey", "safe", "safety", "security", "accept", "acceptance",
    "gentle", "kind", "patient", "compassion", "empathy", "sympathy", "warm",
    "reassure", "reassurance", "encourage", "encouragement", "motivate", "motivation"
]

# Words that indicate harmful/dangerous content
harmful_words = [
    "kill", "killing", "murder", "suicide", "suicidal", "self-harm", "selfharm",
    "harm", "harmful", "dangerous", "abuse", "abusive", "violence", "violent",
    "threat", "threaten", "attack", "assault", "manipulate", "manipulation",
    "gaslight", "gaslighting", "weaponize", "toxic", "narcissist", "narcissistic",
    "exploit", "exploitation", "predator", "predatory", "rape", "rapist", "molest",
    "torture", "murder", "genocide", "terrorism", "terrorist", "bomb", "weapon",
    "drugs", "illegal", "criminal", "crime", "incest", "pedophile", "pedophilia",
    "prostitution", "trafficking", "slavery", "selfish", "cruel", "sadistic"
]

def quality_check(response_text):
    """Check if response contains at least one quality word (case insensitive)"""
    if not response_text:
        return False
    text_lower = response_text.lower()
    for word in quality_words:
        if word in text_lower:
            return True
    return False


def safety_check(response_text):
    """Check if response contains any harmful words (case insensitive)"""
    if not response_text:
        return True  # Empty is safe (will be filtered by quality)
    text_lower = response_text.lower()
    for word in harmful_words:
        if word in text_lower:
            return False  # Contains harmful content
    return True


def filter_quality_safety(data_list):
    """Filter data based on quality and safety checks"""
    quality_filtered = []
    safety_filtered = []
    harm_count = 0
    quality_count = 0
    
    for item in data_list:
        convs = item.get('conversations', [])
        assistant_msg = next((m['content'] for m in convs if m['role'] == 'assistant'), "")
        
        # Safety check first
        if not safety_check(assistant_msg):
            harm_count += 1
            continue
        
        # Quality check
        if quality_check(assistant_msg):
            quality_filtered.append(item)
            quality_count += 1
        else:
            safety_filtered.append(item)
    
    return quality_filtered, harm_count, len(safety_filtered), quality_count


print("✅ Quality & Safety filters defined")

✅ Quality & Safety filters defined


## 11. Apply Filters

Run quality and safety checks on all unique rows:
- First apply **safety filter** (removes harmful content)
- Then apply **quality filter** (keeps therapeutic responses)

**Output**: Count of removed (harmful) and passed (quality + safe) examples


In [76]:
# Apply Quality & Safety filters
print("\n🛡️ Step 2: Applying Quality & Safety filters...")
print("   Safety: Removes responses with harmful words")
print("   Quality: Keeps only responses with therapeutic keywords")

quality_filtered, harm_count, poor_quality_count, good_quality_count = filter_quality_safety(unique_rows)

print(f"   ❌ Removed (harmful content): {harm_count}")
print(f"   ❌ Removed (poor quality - no therapeutic words): {poor_quality_count}")
print(f"   ✅ Passed (quality & safe): {good_quality_count}")


🛡️ Step 2: Applying Quality & Safety filters...
   Safety: Removes responses with harmful words
   Quality: Keeps only responses with therapeutic keywords
   ❌ Removed (harmful content): 25879
   ❌ Removed (poor quality - no therapeutic words): 53296
   ✅ Passed (quality & safe): 229402


## 12. Save Final Dataset

Export the filtered dataset in two formats:

### CSV Format
Columns: `user`, `assistant`, `dataset_name`
- Human-readable, easy to inspect

### JSON Format
Full conversation objects with metadata
- Preserves complete data structure for training

**Location**: Saved to `/kaggle/working/` directory


In [78]:
# Save to Kaggle working directory
import os

csv_rows = []
for item in quality_filtered:
    convs = item.get('conversations', [])
    dataset = item.get('dataset_name', 'unknown')
    user_msg = next((m['content'] for m in convs if m['role'] == 'user'), "")
    assistant_msg = next((m['content'] for m in convs if m['role'] == 'assistant'), "")
    csv_rows.append({'user': user_msg, 'assistant': assistant_msg, 'dataset_name': dataset})

df = pd.DataFrame(csv_rows)
df.to_csv("psychology_dataset.csv", index=False)

with open("psychology_dataset.json", 'w', encoding='utf-8') as f:
    json.dump(quality_filtered, f, ensure_ascii=False, indent=2)

csv_size = os.path.getsize("psychology_dataset.csv") / (1024*1024)
json_size = os.path.getsize("psychology_dataset.json") / (1024*1024)

print("\n" + "=" * 50)
print("✅ FILES SAVED TO KAGGLE")
print("=" * 50)
print(f"psychology_dataset.csv: {csv_size:.2f} MB ({len(quality_filtered)} rows)")
print(f"psychology_dataset.json: {json_size:.2f} MB ({len(quality_filtered)} rows)")
print("\n📁 Location: /kaggle/working/")


✅ FILES SAVED TO KAGGLE
psychology_dataset.csv: 91.02 MB (229402 rows)
psychology_dataset.json: 516.93 MB (229402 rows)

📁 Location: /kaggle/working/
